# Konkani ASR Training - 10K Dataset with Dual GPU

Train Konkani ASR model on 10,000 samples using dual GPU for faster training.

## Expected Results:
- Training time: ~9 hours (vs 18 hours on single GPU)
- Should produce actual transcriptions (not blanks)
- Target validation loss: < 2.5

## Hardware:
- Kaggle P100 x2 GPUs
- DataParallel for multi-GPU training

## Step 1: Setup Environment

In [ ]:
# Install dependencies
!pip install -q torch torchaudio librosa soundfile jiwer pyyaml tensorboard

In [ ]:
import os
import sys
import json
import torch
import torchaudio
from pathlib import Path
import numpy as np
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Step 2: Check Dataset

In [ ]:
# List available datasets
!ls -lh /kaggle/input/

In [ ]:
# Set paths - UPDATE THIS to match your dataset name
DATA_ROOT = Path('/kaggle/input/konkani-10k-dataset')

# Check structure
print("Dataset structure:")
!ls -lh {DATA_ROOT}

## Step 3: Extract and Prepare Data

In [ ]:
# Extract if zipped
import zipfile

zip_files = list(DATA_ROOT.glob('*.zip'))
if zip_files:
    print(f"Found {len(zip_files)} zip files. Extracting...")
    for zip_file in zip_files:
        print(f"Extracting {zip_file.name}...")
        with zipfile.ZipFile(zip_file, 'r') as zip_ref:
            zip_ref.extractall('/kaggle/working/')
    print("✓ Extraction complete!")
else:
    print("No zip files found, data already extracted")

In [ ]:
# Check extracted data
!ls -lh /kaggle/working/

## Step 4: Prepare Full Dataset (88 hours)

In [ ]:
# Run data preparation script
# This combines KonkaniRawSpeechCorpus (84h) + existing data (21h)
!python /kaggle/working/scripts/prepare_raw_corpus_data.py

In [ ]:
# Verify manifests created
manifest_dir = Path('/kaggle/working/data/konkani-combined/manifests')
if manifest_dir.exists():
    print("✓ Manifests created:")
    for manifest in manifest_dir.glob('*.json'):
        with open(manifest) as f:
            data = json.load(f)
            print(f"  {manifest.name}: {len(data)} samples")
else:
    print("✗ Manifests not found, using existing data")
    manifest_dir = Path('/kaggle/working/data/konkani-asr-v0/splits/manifests')

## Step 5: Configure Training (FIXED SETTINGS)

In [ ]:
# Training configuration with FIXES
import yaml

config = {
    'model': {
        'vocab_size': 200,
        'input_dim': 80,
        'd_model': 256,
        'encoder_layers': 12,
        'decoder_layers': 6,
        'num_heads': 4,
        'conv_kernel_size': 31,
        'dropout': 0.2
    },
    'training': {
        'learning_rate': 0.0003,      # 🔥 Increased from 0.0001
        'weight_decay': 0.0001,
        'grad_clip': 5.0,             # 🔥 Added gradient clipping
        'ctc_weight': 0.8,            # 🔥 CRITICAL FIX: was 0.3
        'batch_size': 2,
        'gradient_accumulation_steps': 4,
        'mixed_precision': True,
        'num_epochs': 100,            # 🔥 More epochs
        'save_every': 5,
        'test_every': 5               # 🔥 Test every 5 epochs
    },
    'data': {
        'train_manifest': str(manifest_dir / 'train.json'),
        'val_manifest': str(manifest_dir / 'val.json'),
        'vocab_file': '/kaggle/working/data/vocab.json',
        'num_workers': 2
    },
    'paths': {
        'checkpoint_dir': '/kaggle/working/checkpoints',
        'log_dir': '/kaggle/working/logs'
    },
    'device': 'cuda'
}

# Save config
os.makedirs('/kaggle/working/config', exist_ok=True)
with open('/kaggle/working/config/training_config_fixed.yaml', 'w') as f:
    yaml.dump(config, f)

print("✓ Training config saved with FIXES:")
print(f"  - CTC weight: {config['training']['ctc_weight']} (was 0.3)")
print(f"  - Learning rate: {config['training']['learning_rate']} (was 0.0001)")
print(f"  - Gradient clip: {config['training']['grad_clip']} (was None)")
print(f"  - Testing: Every {config['training']['test_every']} epochs")

## Step 6: Start Training with Periodic Testing

In [ ]:
# Start training
!python /kaggle/working/training_scripts/train_konkanivani_asr.py \
    --config /kaggle/working/config/training_config_fixed.yaml \
    --epochs 100

## Step 7: Monitor Progress

### Expected Timeline:
- **Epoch 1-10**: Blank prob 95-98% (learning basics)
- **Epoch 10-20**: Blank prob 80-90% (characters appearing)
- **Epoch 20-40**: Blank prob 50-80% ✅ **WORKING!**
- **Epoch 40-100**: Blank prob 30-50% (refinement)

In [ ]:
# Check test results
test_results_dir = Path('/kaggle/working/checkpoints')
test_files = sorted(test_results_dir.glob('test_results_epoch_*.json'))

if test_files:
    print("Test Results Summary:")
    print("=" * 80)
    for test_file in test_files:
        with open(test_file) as f:
            results = json.load(f)
            epoch = results.get('epoch', '?')
            blank_prob = results.get('avg_blank_prob', 0)
            status = '✅ WORKING!' if blank_prob < 80 else '❌ Not yet'
            print(f"Epoch {epoch:3d}: Blank prob {blank_prob:5.1f}% - {status}")
else:
    print("No test results yet. Check back after epoch 5.")

## Step 8: Download Best Checkpoint

In [ ]:
# Find best checkpoint (lowest validation loss)
checkpoint_dir = Path('/kaggle/working/checkpoints')
checkpoints = sorted(checkpoint_dir.glob('checkpoint_epoch_*.pt'))

if checkpoints:
    best_ckpt = None
    best_val_loss = float('inf')
    
    for ckpt_path in checkpoints:
        ckpt = torch.load(ckpt_path, map_location='cpu')
        val_loss = ckpt.get('val_loss', float('inf'))
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_ckpt = ckpt_path
    
    print(f"Best checkpoint: {best_ckpt.name}")
    print(f"Validation loss: {best_val_loss:.4f}")
    
    # Copy to best_model.pt
    import shutil
    shutil.copy(best_ckpt, checkpoint_dir / 'best_model.pt')
    print("✓ Saved as best_model.pt")
else:
    print("No checkpoints found yet")

In [ ]:
# Create download link
from IPython.display import FileLink

print("Download your trained model:")
FileLink('/kaggle/working/checkpoints/best_model.pt')

## Step 9: Quick Test

In [ ]:
# Test the best model on a few samples
!python /kaggle/working/scripts/test_best_model.py \
    --checkpoint /kaggle/working/checkpoints/best_model.pt \
    --max_files 10

## Summary

### Key Fixes Applied:
1. ✅ CTC weight: 0.3 → 0.8 (critical for transcription)
2. ✅ Learning rate: 0.0001 → 0.0003 (faster learning)
3. ✅ Added gradient clipping: 5.0 (stability)
4. ✅ Full dataset: 21h → 88h (4x more data)
5. ✅ Periodic testing: Monitor every 5 epochs

### Expected Results:
- Model should start working by epoch 20-30
- Blank probability should drop below 80%
- Transcriptions should be recognizable
- Final CER should be 20-40%

### Next Steps:
1. Download best_model.pt
2. Test locally on your audio files
3. Deploy for production use

## Final Step: Generate Training Graphs

Visualize training progress with loss and accuracy curves.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Read training history from TensorBoard logs or saved metrics
# For now, we'll create a simple visualization function

def plot_training_metrics(train_losses, val_losses, save_path='training_metrics.png'):
    """
    Plot training and validation losses
    
    Args:
        train_losses: List of training losses per epoch
        val_losses: List of validation losses per epoch
        save_path: Where to save the plot
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = range(1, len(train_losses) + 1)
    
    # Loss plot
    ax1.plot(epochs, train_losses, 'b-o', label='Training Loss', linewidth=2)
    ax1.plot(epochs, val_losses, 'r-s', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training vs Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Final comparison
    final_train = train_losses[-1]
    final_val = val_losses[-1]
    
    ax2.bar(['Training', 'Validation'], [final_train, final_val], 
            color=['blue', 'red'], alpha=0.7, width=0.6)
    ax2.set_ylabel('Final Loss', fontsize=12)
    ax2.set_title('Final Loss Comparison', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Add value labels on bars
    for i, v in enumerate([final_train, final_val]):
        ax2.text(i, v + 0.05, f'{v:.2f}', ha='center', fontweight='bold')
    
    # Add summary text
    summary_text = f"""Training Summary:
    
Total Epochs: {len(train_losses)}
Training Loss: {train_losses[0]:.2f} → {final_train:.2f}
Validation Loss: {val_losses[0]:.2f} → {final_val:.2f}

Best Val Loss: {min(val_losses):.2f} (Epoch {val_losses.index(min(val_losses))+1})
"""
    
    fig.text(0.98, 0.5, summary_text, 
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
             fontsize=10, verticalalignment='center',
             horizontalalignment='left', family='monospace')
    
    plt.suptitle('Konkani ASR Training Metrics - 10K Dataset (Dual GPU)', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n✓ Graph saved to: {save_path}")

print('✓ Graph function defined')

In [ ]:
# Generate the training graph
# Note: Replace these with actual values from your training loop
# You should save train_losses and val_losses during training

# Example: Load from saved metrics file
try:
    # If you saved metrics during training
    import json
    with open('/kaggle/working/training_metrics.json', 'r') as f:
        metrics = json.load(f)
    train_losses = metrics['train_losses']
    val_losses = metrics['val_losses']
except:
    # Or extract from TensorBoard logs
    from tensorboard.backend.event_processing import event_accumulator
    
    ea = event_accumulator.EventAccumulator('/kaggle/working/logs')
    ea.Reload()
    
    train_losses = [s.value for s in ea.Scalars('train/loss')]
    val_losses = [s.value for s in ea.Scalars('val/loss')]

# Generate the plot
plot_training_metrics(
    train_losses, 
    val_losses, 
    save_path='/kaggle/working/konkani_asr_training_graph.png'
)

print('\n' + '='*80)
print('TRAINING COMPLETE!')
print('='*80)
print(f'Final Training Loss: {train_losses[-1]:.4f}')
print(f'Final Validation Loss: {val_losses[-1]:.4f}')
print(f'Best Validation Loss: {min(val_losses):.4f} (Epoch {val_losses.index(min(val_losses))+1})')
print('='*80)

## Download Results

Download the following files:
- `checkpoints/best_model.pt` - Best model checkpoint
- `konkani_asr_training_graph.png` - Training visualization
- `logs/` - TensorBoard logs